# AI Plant Growth Simulator with Extended Plant Visualization

This notebook builds a simple **plant growth simulator** using:

- a real plant DNA sequence fetched from **NCBI**
- real climate data fetched from **Open-Meteo**
- a small **graph neural network (GCN)** to derive a gene activity signal
- a growth simulation over several days
- extended visualization of plant aspects:
  - height
  - leaf count
  - canopy width
  - root depth
  - stem thickness
  - health index
  - water stress
  - flowering probability
  - biomass
- a simple **2D plant drawing** that changes day by day

## 1) Install and import libraries

In [ ]:
!pip install torch torch-geometric pandas requests matplotlib numpy -q

In [ ]:
import math
from collections import Counter

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, Circle

import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

plt.rcParams["figure.figsize"] = (9, 5)

## 2) Download a real plant DNA sequence from NCBI

We use the **Arabidopsis thaliana chloroplast genome** as a real biological example.

In [ ]:
ncbi_url = (
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    "?db=nuccore&id=NC_000932.1&rettype=fasta&retmode=text"
)

response = requests.get(ncbi_url, timeout=30)
response.raise_for_status()

fasta_text = response.text
dna_sequence = "".join(
    line.strip() for line in fasta_text.splitlines()
    if not line.startswith(">")
)

print("DNA length:", len(dna_sequence))
print("DNA preview:", dna_sequence[:120] + "...")

## 3) Extract simple DNA features

We compute small **k-mer** frequency features from DNA.

In [ ]:
def kmer_features(seq, k=2):
    kmers = [seq[i:i+k] for i in range(len(seq)-k+1)]
    counts = Counter(kmers)
    total = sum(counts.values())
    return {k: v / total for k, v in counts.items()}

features = kmer_features(dna_sequence, k=2)
features

## 4) Build a tiny gene interaction graph

This is a simplified educational version of a **gene regulatory network**.

In [ ]:
segment_count = 4
segment_size = len(dna_sequence) // segment_count

def feature_vector_from_segment(segment):
    feats = kmer_features(segment, k=2)
    return [
        feats.get("AT", 0.0),
        feats.get("CG", 0.0),
        feats.get("GC", 0.0),
    ]

segments = [
    dna_sequence[i * segment_size:(i + 1) * segment_size]
    for i in range(segment_count)
]

x = torch.tensor(
    [feature_vector_from_segment(segment) for segment in segments],
    dtype=torch.float
)

edge_index = torch.tensor([
    [0, 1, 2, 0, 3, 1],
    [1, 2, 3, 3, 0, 0]
], dtype=torch.long)

print("Gene nodes:", x.shape[0])
print("Node features:")
print(x)

## 5) Define and run a simple GCN model

In [ ]:
class GRNModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(3, 8)
        self.conv2 = GCNConv(8, 4)
        self.conv3 = GCNConv(4, 1)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.conv3(x, edge_index)
        return x

model = GRNModel()
gene_activity = model(x, edge_index)

print("Predicted gene activity:")
print(gene_activity)
print("Mean gene activity:", float(gene_activity.mean()))

## 6) Download real climate data from Open-Meteo

Example location: **Tunis**.

In [ ]:
latitude = 36.8065
longitude = 10.1815

weather_url = (
    "https://api.open-meteo.com/v1/forecast"
    f"?latitude={latitude}&longitude={longitude}"
    "&past_days=9&forecast_days=1"
    "&hourly=temperature_2m,relative_humidity_2m,shortwave_radiation"
    "&timezone=auto"
)

response = requests.get(weather_url, timeout=30)
response.raise_for_status()
weather_json = response.json()

hourly = pd.DataFrame({
    "time": pd.to_datetime(weather_json["hourly"]["time"]),
    "temperature": weather_json["hourly"]["temperature_2m"],
    "humidity": weather_json["hourly"]["relative_humidity_2m"],
    "sunlight": weather_json["hourly"]["shortwave_radiation"],
})

climate = (
    hourly.assign(day=hourly["time"].dt.date)
    .groupby("day", as_index=False)
    .agg(
        temperature=("temperature", "mean"),
        humidity=("humidity", "mean"),
        sunlight=("sunlight", "mean")
    )
)

climate["day_index"] = np.arange(1, len(climate) + 1)
climate

## 7) Simulate plant growth over time

The growth rate combines:

- gene activity
- temperature effect
- humidity effect
- sunlight effect

In [ ]:
growth = []
height = 5.0

gene_factor = float(gene_activity.mean())
gene_factor = max(gene_factor, 0.05)

for _, row in climate.iterrows():
    temp_effect = max(row.temperature, 0) / 25
    humidity_effect = row.humidity / 60
    sunlight_effect = row.sunlight / 200

    growth_rate = (
        0.35 * temp_effect +
        0.25 * humidity_effect +
        0.40 * sunlight_effect
    )
    growth_rate *= gene_factor

    height += growth_rate
    growth.append(height)

climate["simulated_height"] = growth
climate

## 8) Generate additional plant aspects to visualize

We enrich the simulation with more visual plant traits.

In [ ]:
climate["leaf_count"] = np.maximum((climate["simulated_height"] * 1.6).astype(int), 2)
climate["canopy_width"] = 1.5 + climate["simulated_height"] * 0.25
climate["stem_thickness"] = 0.08 + climate["simulated_height"] * 0.015
climate["root_depth"] = 1.5 + climate["simulated_height"] * 0.30

temp_score = 1 - (abs(climate["temperature"] - 22) / 15)
humidity_score = 1 - (abs(climate["humidity"] - 60) / 40)
sunlight_score = climate["sunlight"] / climate["sunlight"].max()

climate["health_index"] = (
    0.4 * temp_score.clip(0, 1) +
    0.3 * humidity_score.clip(0, 1) +
    0.3 * sunlight_score.clip(0, 1)
).clip(0, 1)

climate["water_stress"] = (1 - climate["humidity"] / 100).clip(0, 1)

h_min = climate["simulated_height"].min()
h_max = climate["simulated_height"].max()
climate["flowering_probability"] = (
    (climate["simulated_height"] - h_min) /
    (h_max - h_min + 1e-6)
).clip(0, 1)

climate["biomass"] = (
    climate["simulated_height"] *
    climate["canopy_width"] *
    climate["health_index"]
)

climate

## 9) Plot the main growth curve

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(climate["day_index"], climate["simulated_height"], marker="o")
plt.xlabel("Day")
plt.ylabel("Plant Height")
plt.title("Simulated Plant Growth with Real Internet Data")
plt.grid(True)
plt.show()

## 10) Plot the new plant features

In [ ]:
def plot_feature(column_name, y_label, title):
    plt.figure(figsize=(8, 4))
    plt.plot(climate["day_index"], climate[column_name], marker="o")
    plt.xlabel("Day")
    plt.ylabel(y_label)
    plt.title(title)
    plt.grid(True)
    plt.show()

plot_feature("leaf_count", "Leaf Count", "Simulated Leaf Count Over Time")
plot_feature("canopy_width", "Canopy Width", "Simulated Canopy Width Over Time")
plot_feature("root_depth", "Root Depth", "Simulated Root Depth Over Time")
plot_feature("stem_thickness", "Stem Thickness", "Simulated Stem Thickness Over Time")
plot_feature("health_index", "Health Index", "Plant Health Index Over Time")
plot_feature("water_stress", "Water Stress", "Plant Water Stress Over Time")
plot_feature("flowering_probability", "Flowering Probability", "Flowering Probability Over Time")
plot_feature("biomass", "Biomass", "Estimated Plant Biomass Over Time")

## 11) Draw a simplified 2D plant for one day

This function transforms the numeric simulation into a visual plant.

In [ ]:
def draw_plant(row, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 7))

    ax.clear()

    # Ground line
    ax.axhline(0, linewidth=2)

    height = float(row["simulated_height"])
    leaf_count = int(row["leaf_count"])
    stem_thickness = float(row["stem_thickness"])
    root_depth = float(row["root_depth"])
    health = float(row["health_index"])
    flower_prob = float(row["flowering_probability"])

    stem_alpha = 0.5 + 0.5 * health

    # Stem
    ax.plot(
        [0, 0], [0, height],
        linewidth=max(1.0, 8 * stem_thickness),
        alpha=stem_alpha
    )

    # Roots
    root_lines = max(3, min(6, int(round(root_depth))))
    for i in range(root_lines):
        spread = 0.3 + i * 0.15
        ax.plot([0, -spread], [0, -root_depth * 0.3], linewidth=1.5, alpha=0.7)
        ax.plot([0, spread], [0, -root_depth * 0.3], linewidth=1.5, alpha=0.7)

    # Leaves
    usable_leaf_count = min(leaf_count, 16)
    if usable_leaf_count > 0:
        leaf_positions = np.linspace(height * 0.2, height * 0.9, usable_leaf_count)

        for idx, y in enumerate(leaf_positions):
            side = -1 if idx % 2 == 0 else 1
            leaf_width = 0.6 + 0.4 * health
            leaf_height = 0.25 + 0.12 * health

            leaf = Ellipse(
                (side * 0.6, y),
                width=leaf_width,
                height=leaf_height,
                angle=35 * side,
                alpha=0.35 + 0.6 * health
            )
            ax.add_patch(leaf)

            ax.plot([0, side * 0.35], [y, y], linewidth=1.2)

    # Flower
    if flower_prob > 0.7:
        flower_center_y = height + 0.4
        for angle in np.linspace(0, 2 * np.pi, 6, endpoint=False):
            petal_x = 0.22 * np.cos(angle)
            petal_y = flower_center_y + 0.22 * np.sin(angle)
            petal = Circle((petal_x, petal_y), 0.12, alpha=0.7)
            ax.add_patch(petal)

        center = Circle((0, flower_center_y), 0.09, alpha=0.9)
        ax.add_patch(center)

    ax.set_title(f"Plant Visualization - Day {int(row['day_index'])}")
    ax.set_xlim(-3, 3)
    ax.set_ylim(-max(2, root_depth), height + 2)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("auto")
    plt.tight_layout()

## 12) Display one simulated plant

In [ ]:
fig, ax = plt.subplots(figsize=(5, 7))
draw_plant(climate.iloc[-1], ax=ax)
plt.show()

## 13) Display the plant day by day

In [ ]:
for i in range(len(climate)):
    fig, ax = plt.subplots(figsize=(5, 7))
    draw_plant(climate.iloc[i], ax=ax)
    plt.show()

## 14) Display all days in a grid

In [ ]:
n = len(climate)
cols = 3
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 5))
axes = np.array(axes).reshape(-1)

for i in range(n):
    draw_plant(climate.iloc[i], ax=axes[i])

for j in range(n, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

## 15) Export the final simulation table (optional)

In [ ]:
output_path = "plant_simulation_extended_results.csv"
climate.to_csv(output_path, index=False)
print(f"Saved results to: {output_path}")